# 05 - Gold Vendas Diárias - Bike Store Medallion
Notebook de agregação da camada Gold focado em vendas diárias.

Fonte utilizada da camada Silver:
- `bike_store.silver.orders`

## Transformações aplicadas
- Filtro: apenas pedidos com status 'Delivered' e estado 'NY'
- Agregação: soma total de vendas por dia

In [0]:
# Importação de bibliotecas e dados da tabela silver_orders para dataframe
from pyspark.sql.functions import col, sum as sum, round as rnd

df_orders = spark.read.table("bike_store.silver.orders")

df_gold_vendas = df_orders \
    .filter(
        (col("order_status") == "Delivered")
    )

display(df_gold_vendas)

In [0]:
# Criando dataframe de clientes a partir da tabela silver_clients
df_clients = spark.read.table("bike_store.silver.clients")
# Filtro pelo estado = NY
df_clients_ny = df_clients.filter(col("state") == "NY")
display(df_clients_ny)


In [0]:
df_clients_filtered = df_clients_ny \
    .filter(col("state") == "NY") \
    .select("customer_id", "state")

In [0]:
# Join e agregação
df_gold_vendas = df_orders \
    .filter(col("order_status") == "Delivered") \
    .join(df_clients_filtered, on="customer_id", how="inner") \
    .groupBy("order_date") \
    .agg(
        rnd(sum("total_value"), 2).alias("total_vendas")
    ) \
    .orderBy("order_date")

display(df_gold_vendas)

In [0]:
# Salvando os dados na tabela Gold
df_gold_vendas.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("bike_store.gold.vendas_diarias_ny")

print("✅ bike_store.gold.vendas_diarias salvo com sucesso!")